# Brain Tumor Profiler: Kaggle training notebook (Phases 2–4)

**Settings:** Accelerator → *GPU T4 x2* (or P100) · Internet → *On* (pretrained encoder download).

**Inputs:**
1. Add the BraTS 2021 training data as a Kaggle input. Use a mirror whose licence you have checked, and cite the BraTS papers.
2. Add this repo, either by cloning it below or by uploading it as a private Kaggle dataset.

Budget: preprocessing ≈ 10–15 min for 120 patients; training ≈ 1–2 h. `last.pt` is saved every epoch, so if the session dies, re-run the train cell and it resumes.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
REPO_URL = "https://github.com/<your-org>/brain-tumor-profiler.git"   # <-- edit
BRATS_DIR = "/kaggle/input/brats-2021-task1/BraTS2021_Training_Data"    # <-- edit to your input path
WORK = "/kaggle/working/brain-tumor-profiler"

In [ ]:
import os

if not os.path.exists(WORK):
    !git clone -q {REPO_URL} {WORK}
%cd {WORK}
!pip install -q "monai[nibabel,skimage,einops]>=1.4" SimpleITK pydicom reportlab
!pip install -q -e .

## Phase 2: data (patient-level split, cached slice stacks)

In [ ]:
!python scripts/prepare_data.py data.raw_dir={BRATS_DIR} data.num_patients=120     data.processed_dir=/kaggle/working/processed     data.manifest=/kaggle/working/processed/manifest.csv data.splits=/kaggle/working/processed/splits.json

In [ ]:
import json

import pandas as pd

m = pd.read_csv('/kaggle/working/processed/manifest.csv'); print(m.describe())
print({k: len(v) for k, v in json.load(open('/kaggle/working/processed/splits.json')).items()})

## Phase 3: baseline (pretrained FlexibleUNet, T1ce + FLAIR)

In [ ]:
DATA = ("data.processed_dir=/kaggle/working/processed data.manifest=/kaggle/working/processed/manifest.csv "
        "data.splits=/kaggle/working/processed/splits.json")
!python scripts/train.py {DATA} train.output_dir=/kaggle/working/runs/baseline train.epochs=40 train.batch_size=16

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

h = pd.read_csv('/kaggle/working/runs/baseline/metrics.csv')
ax = h.plot(x='epoch', y=['train_loss', 'val_dice_mean'], secondary_y='val_dice_mean', figsize=(8, 3)); plt.show()

## Phase 4: ablations (one run each; budget ~10 runs total)
Uncomment one at a time. Compare **val** Dice only. The test set is used once, at the very end.

In [ ]:
# !python scripts/train.py {DATA} model.pretrained=false train.output_dir=/kaggle/working/runs/scratch
# !python scripts/train.py {DATA} train.lr=1e-4 train.output_dir=/kaggle/working/runs/lr1e-4
# !python scripts/train.py {DATA} data.tumor_to_empty_ratio=0.5 train.output_dir=/kaggle/working/runs/ratio05
# !python scripts/train.py {DATA} model.backbone=efficientnet-b2 train.output_dir=/kaggle/working/runs/b2

## Final test evaluation (run ONCE with the chosen model)

In [ ]:
BEST = "/kaggle/working/runs/baseline/best.pt"
!python scripts/evaluate.py {DATA} --weights {BEST} --split test --out /kaggle/working/runs/final_test
from IPython.display import Image, display

display(Image('/kaggle/working/runs/final_test/boxplot.png')); display(Image('/kaggle/working/runs/final_test/worst_cases.png'))

## Optional (Option C): MONAI BraTS bundle zero-shot reference

In [ ]:
# !pip install -q "monai[fire]"
# !python scripts/bundle_reference.py data.raw_dir={BRATS_DIR} --out /kaggle/working/runs/bundle_reference

Download `runs/<chosen>/best.pt` and `runs/final_test/summary.json`, then run `scripts/make_demo_assets.py` locally.